# Дообучение EEGPT на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-22 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить EEGPT (NeurIPS 2024, код github.com/BINE022/EEGPT) по той же схеме, что REVE, CBraMod, BIOT и LUNA, сравнить и собрать в ансамбль с REVE |
| **Веса** | HF `braindecode/eegpt-pretrained` (`model.safetensors`, кодировщик `target_encoder`, 25,3 млн параметров, BSD-3). Сверены с независимой копией HF `eeg-telecom-paris/eegpt-large-official` (раздел 2). **Предобучение: PhysioNetMI, HGD, TSU, SEED, M3CV — TUH там нет**, поэтому на TUEP не нужно исключать пересечение пациентов (в отличие от REVE, CBraMod, LUNA). Дообученные веса никуда не сохраняются |
| **Данные** | Обучение: NMT train (`mhmdrdwn/nmt-eeg`), до 20 окон по 4 с на запись (те же 80 с сигнала, что у REVE, BIOT, LUNA), 15 % записей — выбор скорости обучения, эпохи и порога. Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`) |
| **Вход** | Как при предобучении авторов (`datasets/pretrain/prepare_pretrain_dataset.py`): 256 Гц, окна 4 с (16 патчей по 64 отсчёта), единицы — мВ (В·1000), средний референт; 19 наших каналов в порядке списка 58 каналов авторов (T3/T4/T5/T6 → T7/T8/P7/P8). Фильтр 0,1–75 Гц + режекция 50 Гц — как в их `make_TUAB.py`. Масштаб NMT в файлах неверный, поэтому каждое окно приводится к общему СКО 15 мкВ по всем каналам (как у CBraMod): соотношение каналов остаётся, абсолютная амплитуда пропадает |
| **Аугментации** | зеркальная замена полушарий (p = 0,5), шум 0,1·СКО, зануление 1–2 каналов (p = 0,5), сдвиг до 2 с — как у LUNA и REVE |
| **Обучение** | полное дообучение: кодировщик + голова зондирования авторов (`linear_probe_EEGPT_*.py`: масштаб каналов, 2048 → 16 на патч, 256 → 1), AdamW (wd 0,05), послойное снижение скорости 0,9 (как в их TUAB-скрипте), разогрев 5 % + косинус, пакет 64, fp16 с проверкой NaN. Базовая скорость — по отложенным 15 % NMT train из 5·10⁻⁵ / 1,5·10⁻⁴ / 5·10⁻⁴ (5·10⁻⁴ — значение авторов для TUAB). **Эпохи — до плато**: ранняя остановка, если отложенный ROC AUC не растёт 2 эпохи подряд (не больше 10 эпох) |
| **Среда** | Kaggle: чтение данных — сеанс без GPU (`PREP_SAVE = True`), обучение — GPU T4 |

In [ ]:
import os, sys, glob, time, subprocess, warnings
from fractions import Fraction
from functools import partial
ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = False       # True — короткая проверка на процессоре (часть записей, 1 эпоха), тестовые цифры не смотреть
PREP_SAVE = False   # True — сеанс без GPU: сверка весов, чтение данных и сохранение окон (разделы 2–3), без обучения
EEGPT_COMMIT = "ea7282f698c406cd423ccbedbe9960853761144c"   # BINE022/EEGPT — код, по которому сверялись модель и подготовка сигнала
if ON_KAGGLE:
    if not os.path.exists("/kaggle/temp/EEGPT"):
        subprocess.run(["git", "clone", "-q", "https://github.com/BINE022/EEGPT.git", "/kaggle/temp/EEGPT"], check=True)
        subprocess.run(["git", "-C", "/kaggle/temp/EEGPT", "checkout", "-q", EEGPT_COMMIT], check=True)
    EEGPT_DIR, OUT, HF_CACHE = "/kaggle/temp/EEGPT", "/kaggle/working", "/kaggle/temp/hf"   # код и веса не в /kaggle/working — не попадут в вывод
else:
    EEGPT_DIR, OUT, HF_CACHE = os.environ["EEGPT_DIR"], os.environ.get("EEGPT_OUT", "."), os.environ.get("HF_CACHE")
sys.path.insert(0, f"{EEGPT_DIR}/downstream")
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from safetensors.torch import load_file
from Modules.models.EEGPT_mcae import EEGTransformer, CHANNEL_DICT   # кодировщик из downstream-скриптов авторов
from Modules.Network.utils import LinearWithConstraint
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_NB = time.time()

SFREQ, SEG, PATCH = 256, 4, 64
L = SFREQ * SEG                 # 1024 отсчёта = 16 патчей по 64, как при предобучении (58 каналов, 256 Гц, 4 с)
CAP_TRAIN, CAP_EVAL = 20, 150   # окон по 4 с на запись: те же 80 с / 600 с сигнала, что у REVE, BIOT и LUNA

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
# 58 каналов предобучения — use_channels_names в datasets/pretrain/prepare_pretrain_dataset.py
USE58 = ["FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8",
         "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8",
         "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
         "PO7", "PO3", "POZ", "PO4", "PO8", "O1", "OZ", "O2"]
NEW_NAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}   # старые названия 10-20 -> 10-10, как в списке авторов
EL_UP = [NEW_NAME.get(e, e).upper() for e in EL]
USE = [c for c in USE58 if c in EL_UP]                        # наши 19 каналов в порядке списка авторов
ORDER = [EL_UP.index(c) for c in USE]                         # перестановка наших каналов в этот порядок
CHANS_ID = torch.tensor([CHANNEL_DICT[c] for c in USE])       # строки таблицы каналов кодировщика (62 строки)
assert len(USE58) == 58 and len(USE) == 19 and all(c in USE58 for c in EL_UP)
print("каналы в порядке авторов:", " ".join(USE))
print("их места среди 58:", [USE58.index(c) for c in USE], "| строки таблицы каналов:", CHANS_ID.tolist())
MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]      # на 19 исходных каналах, до перестановки

# EEGPT large, как в скриптах авторов (finetune_EEGPT_SleepEDF.py, linear_probe_EEGPT_*.py)
ENC = dict(patch_size=PATCH, embed_dim=512, embed_num=4, depth=8, num_heads=8, mlp_ratio=4.0, qkv_bias=True, drop_rate=0.0,
           attn_drop_rate=0.0, drop_path_rate=0.0, init_std=0.02, norm_layer=partial(nn.LayerNorm, eps=1e-6))
DEPTH = ENC["depth"]

class EEGPTCls(nn.Module):
    """Как linear_probe_EEGPT_*.py авторов: обучаемый масштаб каналов -> кодировщик (16 патчей x 4 итоговых токена x 512)
    -> Dropout 0,5 -> 2048 -> 16 на патч -> 256 -> 1 выход. Здесь обучается вся модель (у авторов при зондировании кодировщик заморожен)."""
    def __init__(self, enc_sd):
        super().__init__()
        self.chan_scale = nn.Parameter(torch.ones(1, len(USE), 1) + 0.001 * torch.rand(1, len(USE), 1))
        self.target_encoder = EEGTransformer(img_size=[len(USE), L], **ENC)
        self.target_encoder.load_state_dict(enc_sd, strict=True)      # строго: ошибка при любом пропущенном или лишнем ключе
        self.drop = nn.Dropout(0.5)
        self.linear_probe1 = LinearWithConstraint(4 * 512, 16, max_norm=1)
        self.linear_probe2 = LinearWithConstraint(16 * (L // PATCH), 1, max_norm=0.25)
        self.register_buffer("chans_id", CHANS_ID.clone().unsqueeze(0))
    def forward(self, x):
        z = self.target_encoder(x * self.chan_scale, self.chans_id)   # (B, 16, 4, 512)
        h = self.linear_probe1(self.drop(z.flatten(2)))               # (B, 16, 16)
        return self.linear_probe2(h.flatten(1)).squeeze(-1)

STD_UV, CLIP, NOISE = 15.0, 15.0, 0.1
def batch_input(W, g=None):
    """19 каналов -> (зеркало) -> средний референт (как у авторов) -> общее СКО окна 15 мкВ, обрезка ±15 СКО -> мВ
    (единицы предобучения: В·1000) -> порядок каналов авторов -> (шум, зануление, сдвиг)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    if g is not None:
        m = torch.from_numpy(g.random(len(x)) < 0.5).to(DEV)
        x[m] = x[m][:, MIRROR]
    x = x - x.mean(1, keepdim=True)
    x = (x / (x.std(dim=(1, 2), keepdim=True) + 1e-6)).clamp(-CLIP, CLIP) * (STD_UV / 1000.0)
    x = x[:, ORDER]
    if g is not None:
        x = x + NOISE * (STD_UV / 1000.0) * torch.randn_like(x)
        for b in range(len(x)):
            if g.random() < 0.5:
                x[b, g.choice(len(USE), g.integers(1, 3), replace=False)] = 0
            x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")
def predict(model, X, off, bs=512):
    """Медиана вероятности по окнам каждой записи; окна всех записей идут общими пакетами, off — границы записей."""
    model.eval(); p = []
    with torch.no_grad(), torch.autocast(**AMP):
        for b in range(0, len(X), bs):
            p.append(torch.sigmoid(model(batch_input(X[b:b + bs])).float()).cpu().numpy())
    p = np.concatenate(p)
    return np.array([np.median(p[off[i]:off[i + 1]]) for i in range(len(off) - 1)])

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.1, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def eg_filter(X):
    """0,1-75 Гц + режекция 50 Гц, как в make_TUAB.py авторов."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def to_256(X, sf):
    fr = Fraction(SFREQ, int(round(sf))).limit_denominator(1000)
    return X if fr == 1 else ss.resample_poly(X, fr.numerator, fr.denominator, axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)

def take(X, m):
    """Окна записей из таблицы m (столбцы start, n) подряд + границы записей."""
    idx = np.concatenate([np.arange(s, s + n) for s, n in zip(m["start"], m["n"])])
    return X[idx], np.concatenate([[0], np.cumsum(m["n"].to_numpy())])
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. Сверка весов: HF braindecode ↔ HF telecom-paris ↔ код авторов

Оригинальный чекпоинт `eegpt_mcae_58chs_4s_large4E.ckpt` лежит на figshare по закрытой ссылке, из скриптов она не открывается (ответ 202 — проверка на робота), поэтому сравниваются две независимые копии на HF: braindecode (только `target_encoder`) и telecom-paris (полный чекпоинт: `encoder`, `target_encoder`, `predictor`, `reconstructor`). Авторы для дообучения берут именно `target_encoder.*` (`finetune_EEGPT_SleepEDF.py`, `linear_probe_EEGPT_*.py`).

In [ ]:
import json, requests
from huggingface_hub import hf_hub_download
HF = {"braindecode": ("braindecode/eegpt-pretrained", "model.safetensors", "e41cb3ae2ce4fd9eb736862292c91f8128d15618"),
      "telecom-paris": ("eeg-telecom-paris/eegpt-large-official", "weights.safetensors", "11e0075c388e282a2762496f7f3775652c9e3db9")}
W = {k: load_file(hf_hub_download(r, f, revision=v, cache_dir=HF_CACHE)) for k, (r, f, v) in HF.items()}
sub = lambda d, p: {k[len(p):]: v for k, v in d.items() if k.startswith(p)}
for k, d in W.items():
    pre = pd.Series([n.split(".")[0] for n in d]).value_counts().to_dict()
    print(f"{k}: тензоров {len(d)}, параметров {sum(v.numel() for v in d.values()):,} | по частям: {pre}")
A, B = sub(W["braindecode"], "target_encoder."), sub(W["telecom-paris"], "target_encoder.")
same_keys = set(A) == set(B)
same_shape = same_keys and all(A[k].shape == B[k].shape and A[k].dtype == B[k].dtype for k in A)
bitwise = same_shape and all(torch.equal(A[k], B[k]) for k in A)
print(f"target_encoder: ключей {len(A)} / {len(B)} | ключи совпадают: {same_keys} | формы и типы: {same_shape} | значения побитово: {bitwise}")
E = sub(W["telecom-paris"], "encoder.")
enc_vs_target = max((E[k].float() - B[k].float()).abs().max().item() for k in B)
print(f"telecom-paris: encoder против target_encoder (EMA-копия), макс. разница {enc_vs_target:.3g} — для дообучения берётся target_encoder, как у авторов")
chans_ok = torch.equal(W["braindecode"]["chans_id"].flatten(), torch.arange(62))
print("chans_id в файле braindecode = 0..61 (вся таблица каналов):", chans_ok)

# строгая загрузка в класс авторов: и с 58 каналами предобучения, и с нашими 19
rows = []
for n_ch in (58, 19):
    m = EEGTransformer(img_size=[n_ch, L], **ENC)
    r = m.load_state_dict(A, strict=True)
    n_par = sum(p.numel() for p in m.parameters())
    with torch.no_grad():
        z = m(torch.randn(2, n_ch, L) * 0.015, CHANS_ID.unsqueeze(0) if n_ch == 19 else torch.tensor([[CHANNEL_DICT[c] for c in USE58]]))
    print(f"код авторов, {n_ch} каналов: strict=True — пропущено {len(r.missing_keys)}, лишних {len(r.unexpected_keys)}, "
          f"параметров {n_par:,} (в файле {sum(v.numel() for v in A.values()):,}) | выход {tuple(z.shape)}, конечный: {bool(torch.isfinite(z).all())}")
    rows.append(dict(check=f"strict-загрузка в EEGTransformer авторов ({n_ch} каналов)", ok=not r.missing_keys and not r.unexpected_keys and n_par == sum(v.numel() for v in A.values())))
del m

# класс braindecode.models.EEGPT (по заданию — если braindecode нет, ставится без зависимостей)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "braindecode==1.3.1"], check=True, capture_output=True)
    for _ in range(25):  # недостающие модули ставятся по одному без зависимостей, чтобы не трогать torch
        try:
            from braindecode.models import EEGPT as BD_EEGPT; break
        except ModuleNotFoundError as e:
            print("  ставлю", e.name); subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", e.name.split(".")[0].replace("_", "-")], check=True, capture_output=True)
            for k in [k for k in sys.modules if k.startswith("braindecode")]: del sys.modules[k]
    cfg = json.load(open(hf_hub_download(HF["braindecode"][0], "config.json", revision=HF["braindecode"][2], cache_dir=HF_CACHE)))
    cfg.pop("braindecode_version", None)
    mb = BD_EEGPT(**cfg)
    r = mb.load_state_dict(W["braindecode"], strict=False)
    enc_missing = [k for k in r.missing_keys if k.startswith("target_encoder.")]
    print(f"braindecode.models.EEGPT: нет в файле {len(r.missing_keys)} ключей {r.missing_keys[:6]} (из них кодировщика: {len(enc_missing)}), "
          f"лишних в файле {len(r.unexpected_keys)} {r.unexpected_keys[:6]}")
    rows.append(dict(check="файл braindecode грузится в braindecode.models.EEGPT: весь кодировщик, без лишних ключей",
                     ok=not r.unexpected_keys and not enc_missing))
    del mb
except Exception as e:
    print("braindecode.models.EEGPT не проверен:", repr(e)[:300]); rows.append(dict(check="braindecode.models.EEGPT", ok=None))
try:
    st = requests.get("https://figshare.com/s/e37df4f8a907a866df4b", timeout=20).status_code
except Exception as e:
    st = repr(e)[:80]
print("figshare (оригинальный чекпоинт), ответ:", st)
rows = [dict(check="ключи target_encoder braindecode = telecom-paris", ok=same_keys), dict(check="формы и типы совпадают", ok=same_shape),
        dict(check="значения совпадают побитово", ok=bitwise), dict(check="chans_id braindecode = 0..61", ok=chans_ok)] + rows + \
       [dict(check=f"figshare доступен из скрипта (ответ {st})", ok=st == 200)]
pd.DataFrame(rows).to_csv(f"{OUT}/eegpt_weights_check.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))
assert same_keys and same_shape and bitwise, "веса на HF расходятся"
SD = {k: v.clone() for k, v in A.items()}
del W, A, B, E
def new_eegpt(seed):
    torch.manual_seed(seed)
    return EEGPTCls(SD).to(DEV)
print("EEGPT для дообучения:", sum(p.numel() for p in new_eegpt(0).parameters()), "параметров")

## 3. NMT и турецкий набор

Чтение выполнялось отдельным сеансом Kaggle без GPU (этот же код, `PREP_SAVE = True`, блокнот `zakharsann/h1-eegpt-prep`): окна сохраняются в float16 и подключаются к GPU-сеансу как вход — GPU не тратится на ~20 мин чтения. Турецкий набор: первые 19 каналов (относительно уха) → средний референт, 500 → 256 Гц, фильтр по всей записи, затем 3 окна по 4 с из середины каждого 15-секундного куска (окна не пересекают стыки кусков).

In [ ]:
import mne
mne.set_log_level("ERROR")
t0 = time.time()
PREP_IN = sorted(glob.glob("/kaggle/input/**/eegpt_prep_nmt_meta.csv", recursive=True))
if PREP_IN and not PREP_SAVE:
    P = os.path.dirname(PREP_IN[0])
    lab, tmeta = pd.read_csv(f"{P}/eegpt_prep_nmt_meta.csv"), pd.read_csv(f"{P}/eegpt_prep_tr_meta.csv")
    NMT_X, TR_X = np.load(f"{P}/eegpt_prep_nmt.npy"), np.load(f"{P}/eegpt_prep_tr.npy")
    print(f"готовые окна: NMT {NMT_X.shape}, Турция {TR_X.shape} | {time.time()-t0:.0f} с")
    if SMOKE:
        lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                         lab[lab["loc"] == "eval"].head(20)]).sort_values("recordname").reset_index(drop=True)
        tmeta = pd.concat([tmeta.head(10), tmeta.tail(10)]).reset_index(drop=True)
else:
    ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
    ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
    lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
    paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
    lab["path"] = lab["recordname"].map(paths)
    lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
    lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
    tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
             [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
    if SMOKE:
        lab = pd.concat([lab[(lab["loc"] == "train") & (lab.y == 0)].head(40), lab[(lab["loc"] == "train") & (lab.y == 1)].head(20),
                         lab[(lab["loc"] == "eval")].head(20)]).sort_values("recordname").reset_index(drop=True)
        tfiles = tfiles[:10] + tfiles[-10:]
    print(pd.crosstab(lab["loc"], lab["label"]))

    def warm(p):
        with open(p, "rb") as f:
            while f.read(1 << 24):
                pass
    allf = list(lab.path) + [f for f, _ in tfiles]
    with ThreadPoolExecutor(32) as ex:
        for i, _ in enumerate(ex.map(warm, allf)):
            if (i + 1) % 500 == 0 or i + 1 == len(allf):
                print(f"подкачано {i+1}/{len(allf)} {time.time()-t0:.0f} с", flush=True)

    ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
    TARGET = [e.upper() for e in EL]
    def norm(ch):
        c = ch.upper().replace("EEG", "").strip()
        for s in ["-REF", "-AVG", "-LE", "-AR"]:
            c = c.replace(s, "")
        return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

    NMT_STORE_K = 100.0   # масштаб в файлах неверный; каждое окно всё равно приводится к СКО 15 мкВ, множитель только для float16
    def nmt_job(i):
        r = lab.iloc[i]
        try:
            raw = mne.io.read_raw_edf(r.path, preload=True)
            names = {norm(c): c for c in raw.ch_names}
            X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
            X = to_256(X, raw.info["sfreq"])
        except Exception:
            return r.recordname, None
        W = windows(eg_filter(X))
        if len(W) == 0:
            return r.recordname, None
        if r["loc"] == "train" and len(W) > CAP_TRAIN:
            W = W[np.sort(np.random.default_rng(i).choice(len(W), CAP_TRAIN, replace=False))]
        elif r["loc"] == "eval":
            W = W[spread(len(W), CAP_EVAL)]
        return r.recordname, W.astype(np.float16)

    NMT = {}
    with ThreadPoolExecutor(4) as ex:
        for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
            if res is not None:
                NMT[name] = res
            if (i + 1) % 400 == 0 or i + 1 == len(lab):
                print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
    lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)[["recordname", "loc", "label", "y"]]
    lab["n"] = [len(NMT[n]) for n in lab.recordname]; lab["start"] = np.concatenate([[0], np.cumsum(lab.n.to_numpy())[:-1]])
    NMT_X = np.concatenate([NMT[n] for n in lab.recordname]); del NMT

    # индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
    IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]
    def tr_job(f):
        d = scipy.io.loadmat(f)["data"]
        X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
        X = X - X.mean(1, keepdims=True)          # средний референт — для СКО записи (оценка перекоса), как в образцах
        X = to_256(X, 500)                        # кусок 15 с = 3840 отсчётов
        k, T = X.shape[0], X.shape[-1]
        Xf = eg_filter(np.concatenate(list(X), axis=-1)).reshape(19, k, T).transpose(1, 0, 2)
        o = (T - 3 * L) // 2                      # 3 окна по 4 с из середины куска
        W = Xf[:, :, o:o + 3 * L].reshape(k, 19, 3, L).transpose(0, 2, 1, 3).reshape(-1, 19, L)
        return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))
    TRW, amp = [], []
    with ThreadPoolExecutor(4) as ex:
        for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
            TRW.append(a); amp.append(b)
    tmeta = pd.DataFrame({"id": [os.path.basename(f) for f, _ in tfiles], "y": [yy for _, yy in tfiles], "amp": amp, "n": [len(w) for w in TRW]})
    tmeta["start"] = np.concatenate([[0], np.cumsum(tmeta.n.to_numpy())[:-1]])
    TR_X = np.concatenate(TRW); del TRW
    print(f"прочитано за {time.time()-t0:.0f} с")
    if PREP_SAVE:
        np.save(f"{OUT}/eegpt_prep_nmt.npy", NMT_X); np.save(f"{OUT}/eegpt_prep_tr.npy", TR_X)
        lab.to_csv(f"{OUT}/eegpt_prep_nmt_meta.csv", index=False); tmeta.to_csv(f"{OUT}/eegpt_prep_tr_meta.csv", index=False)
        print(f"окна сохранены: NMT {NMT_X.shape}, Турция {TR_X.shape}, {(NMT_X.nbytes + TR_X.nbytes)/2**30:.2f} ГБ")

print(pd.crosstab(lab["loc"], lab["label"]))
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
EV_X, EV_OFF = take(NMT_X, ev); TRK_X, TRK_OFF = take(TR_X, tmeta)
y_tr, tr_ids, TR_AMP = tmeta.y.to_numpy(), tmeta.id.tolist(), tmeta.amp.to_numpy()
print("окон по 4 с: NMT train", int(tr.n.sum()), "| NMT eval", len(EV_X), "| Турция", len(TRK_X), f"(на запись: медиана {int(tmeta.n.median())})")
print(f"Турция: эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение EEGPT

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)   # то же деление, что у LUNA
trx = tr.set_index("recordname")
fit_m, val_m = trx.loc[fit_names].reset_index(), trx.loc[val_names].reset_index()
Xw, _ = take(NMT_X, fit_m); yw = np.repeat(fit_m.y.to_numpy(), fit_m.n.to_numpy())
VAL_X, VAL_OFF = take(NMT_X, val_m); y_val = val_m.y.to_numpy()
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор: {len(val_names)} ({len(VAL_X)} окон)")

BS, MAX_EP, PATIENCE, LLRD, WD = 64, (1 if SMOKE else 10), 2, 0.9, 0.05
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()

def lr_scale(n):
    """Послойное снижение скорости 0,9, как layer_decay в run_class_finetuning_EEGPT_change.py авторов."""
    if n.startswith("target_encoder.blocks."):
        return LLRD ** (DEPTH - int(n.split(".")[2]))     # блок i -> 0,9^(8-i)
    if n.startswith(("target_encoder.patch_embed", "target_encoder.chan_embed", "target_encoder.summary_token")):
        return LLRD ** (DEPTH + 1)                         # вход кодировщика
    return 1.0                                             # итоговая норма кодировщика, масштаб каналов, голова

def make_opt(model, total_steps, lr):
    """AdamW; без weight decay — смещения, нормы, итоговые токены и масштаб каналов (как get_parameter_groups авторов)."""
    groups = {}
    for n, p in model.named_parameters():
        no_wd = p.ndim <= 1 or n.endswith((".bias", "summary_token", "chan_scale"))
        groups.setdefault((lr_scale(n), 0.0 if no_wd else WD), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": lr * k, "weight_decay": wd} for (k, wd), ps in groups.items()], betas=(0.9, 0.999))
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

def train_run(seed, lr, deadline, tag="eegpt"):
    """Ранняя остановка по отложенным 15 %: стоп, если ROC AUC не растёт PATIENCE эпох подряд; возвращается лучшая эпоха."""
    g = np.random.default_rng(seed)
    model = new_eegpt(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep, lr)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad, t_ep, stop = -1, None, 0, [], 0, [], "максимум эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        if t_ep and time.time() + np.mean(t_ep) > deadline:
            stop = "время"; break
        te = time.time(); model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = np.sort(idx[b:b + BS])
            l, bad = step(model, opt, sched, scaler, batch_input(Xw[j], g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, VAL_X, VAL_OFF))
        hist.append((round(float(np.nanmean(losses)), 3), round(auc, 3))); t_ep.append(time.time() - te)
        print(f"  {tag} lr {lr:g} запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | "
              f"шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
    model.load_state_dict(best_state)
    return model, dict(best_ep=best_ep, best_auc=best_auc, hist=hist, n_bad=n_bad, stop=stop, epochs=len(hist), min_per_ep=float(np.mean(t_ep)) / 60)

In [ ]:
# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны убывать и уйти ниже 0,69 ---
g0 = np.random.default_rng(123)
n_sm, ep_sm = (32, 3) if SMOKE else (128, 8)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], n_sm, replace=False) for c in (0, 1)])
model = new_eegpt(0); opt, sched = make_opt(model, ep_sm * (len(sm) // BS), 1.5e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
model.train(); sm_losses, tt = [], []
for ep in range(ep_sm):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, scaler, batch_input(Xw[j]), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:])) if len(tt) > 3 else float(np.median(tt))
del model, opt
if DEV == "cuda": torch.cuda.empty_cache()
CONVERGED = bool(np.isfinite(sm_losses).all() and np.mean(sm_losses[-2:]) < np.mean(sm_losses[:2]) - 0.05 and min(sm_losses) < 0.66)
print("сходимость на подвыборке:", "есть" if CONVERGED else f"НЕТ ({np.round(sm_losses, 3)}) — продолжаю, скорость выбирается по отложенным 15 %")
est_ep = (len(yw) / BS * t_step * 1.15 + len(VAL_X) / BS * t_step / 3) / 60   # мин на эпоху (+15 % на аугментации, оценка отложенных)
print(f"шаг {t_step:.3f} с | оценка эпохи {est_ep:.1f} мин")

In [ ]:
# --- выбор базовой скорости обучения: только по отложенным 15 % NMT train (запуск 0) ---
TRAIN_MIN = 62   # бюджет задачи 1,5 ч GPU: готовые окна ~2 мин, сверка весов ~2 мин, итоговые предсказания ~1 мин на запуск
T_TRAIN = time.time(); DEADLINE = T_TRAIN + TRAIN_MIN * 60
EP_GUESS = 5     # ожидаемое число эпох до остановки (уточняется по ходу)
GRID = {}
for lr in ([1.5e-4] if SMOKE else [5e-5, 1.5e-4, 5e-4]):
    ep_run = np.mean([v["epochs"] for v in GRID.values()]) if GRID else EP_GUESS
    per_ep = np.mean([v["min_per_ep"] for v in GRID.values()]) if GRID else est_ep
    need = (1 + 2) * ep_run * per_ep + 3 * 1.5          # этот запуск + два запуска с другими seed + предсказания
    if GRID and (time.time() - T_TRAIN) / 60 + need > TRAIN_MIN:
        print(f"ПРОПУСК lr {lr:g}: не укладывается (нужно ~{need:.0f} мин)"); continue
    model, info = train_run(0, lr, DEADLINE, tag="выбор")
    GRID[lr] = dict(model=model.cpu(), **info)
    print(f"lr {lr:g}: лучшая эпоха {info['best_ep']} из {info['epochs']} ({info['stop']}), отложенные NMT {info['best_auc']:.3f}", flush=True)
LR = max(GRID, key=lambda k: GRID[k]["best_auc"])
print("выбрана скорость:", LR)
pd.DataFrame([dict(lr=k, best_epoch=v["best_ep"], epochs=v["epochs"], stop=v["stop"], val=round(v["best_auc"], 3), hist=v["hist"],
                   nan_steps=v["n_bad"], min_per_epoch=round(v["min_per_ep"], 2)) for k, v in GRID.items()]).to_csv(f"{OUT}/eegpt_lr_choice.csv", index=False)
for k in [k for k in GRID if k != LR]: del GRID[k]["model"]

# --- 3 запуска с выбранной скоростью (запуск 0 — уже обученный при выборе); тестовые наборы — один раз ---
RUNS, dur = [], []
for seed in ([0] if SMOKE else [0, 1, 2]):
    t0 = time.time()
    if seed == 0:
        G = GRID[LR]; model, info = G["model"].to(DEV), {k: v for k, v in G.items() if k != "model"}
    else:
        est = GRID[LR]["epochs"] * GRID[LR]["min_per_ep"] * 60
        if time.time() + est > DEADLINE + 5 * 60:
            print(f"ПРОПУСК запуск {seed}: не укладывается в {TRAIN_MIN} мин"); continue
        model, info = train_run(seed, LR, DEADLINE + 5 * 60)
    s_ev = predict(model, EV_X, EV_OFF); s_tr = predict(model, TRK_X, TRK_OFF); s_val = predict(model, VAL_X, VAL_OFF)
    dur.append(time.time() - t0)
    RUNS.append(dict(model="eegpt", lr=LR, seed=seed, best_epoch=info["best_ep"], epochs=info["epochs"], stop=info["stop"],
                     val=round(info["best_auc"], 3), hist=info["hist"], nan_steps=info["n_bad"],
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr, s_val=s_val,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr))))
    print(f"eegpt запуск {seed}: эпоха {info['best_ep']} из {info['epochs']} ({info['stop']}) | NMT eval {RUNS[-1]['nmt_eval']:.3f} | "
          f"Турция {RUNS[-1]['turkey']:.3f} | медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
          f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)}", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
del GRID
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RUNS])
runs.to_csv(f"{OUT}/eegpt_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги: ROC AUC, пороговые метрики (порог — по отложенным 15 % NMT train, на Турцию перенесён тот же)

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "EEGPT дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "EEGPT дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv(f"{OUT}/eegpt_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))

# пороговые метрики: оценка — среднее вероятностей запусков (у рангов нет общего масштаба между наборами)
p_val = np.mean([r["s_val"] for r in RUNS], 0); p_ev = np.mean([r["s_ev"] for r in RUNS], 0); p_tr = np.mean([r["s_tr"] for r in RUNS], 0)
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "EEGPT", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "NMT eval" else "; перенесён"), "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{OUT}/eegpt_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))

d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
for d, k in [(d1, "s_ev"), (d2, "s_tr")]:
    for r in RUNS: d[f"eegpt_seed{r['seed']}"] = r[k]
d1["eegpt_ens_rank"], d2["eegpt_ens_rank"] = ens_ev, ens_tr
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_m.recordname), "y": y_val})
for r in RUNS: d3[f"eegpt_seed{r['seed']}"] = r["s_val"]
pd.concat([d1, d2, d3]).to_csv(f"{OUT}/eegpt_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 5а. Вывод Kaggle, сеанс 1 (GPU T4, `zakharsann/h1-eegpt-finetune`, версия 1)

Перед ним: сеанс чтения без GPU (`zakharsann/h1-eegpt-prep`, 23 мин чтения) и две короткие проверки на процессоре Kaggle (`SMOKE = True`, без GPU; их тестовые цифры не смотрелись). Тестовые наборы считались один раз.

Скорость 5·10⁻⁵ вышла на плато (лучшая эпоха 7 из 9, остановка по правилу «2 эпохи без роста»). Но эпоха шла 3,9 мин вместо оценённых 3,4, и защита по времени пропустила скорости 1,5·10⁻⁴ и 5·10⁻⁴ и запуски 1–2 — запуски сделаны вторым сеансом (раздел 6).

```
каналы в порядке авторов: FP1 FP2 F7 F3 FZ F4 F8 T7 C3 CZ C4 T8 P7 P3 PZ P4 P8 O1 O2
их места среди 58: [0, 2, 5, 7, 9, 11, 13, 23, 25, 27, 29, 31, 41, 43, 45, 47, 49, 55, 57] | строки таблицы каналов: [0, 2, 7, 9, 11, 13, 15, 25, 27, 29, 31, 33, 43, 45, 47, 49, 51, 59, 61]
устройство: Tesla T4
braindecode: тензоров 103, параметров 25,287,230 | по частям: {'target_encoder': 102, 'chans_id': 1}
telecom-paris: тензоров 413, параметров 101,869,664 | по частям: {'reconstructor': 105, 'predictor': 104, 'encoder': 102, 'target_encoder': 102}
target_encoder: ключей 102 / 102 | ключи совпадают: True | формы и типы: True | значения побитово: True
telecom-paris: encoder против target_encoder (EMA-копия), макс. разница 0 — для дообучения берётся target_encoder, как у авторов
chans_id в файле braindecode = 0..61 (вся таблица каналов): True
код авторов, 58 каналов: strict=True — пропущено 0, лишних 0, параметров 25,287,168 (в файле 25,287,168) | выход (2, 16, 4, 512), конечный: True
код авторов, 19 каналов: strict=True — пропущено 0, лишних 0, параметров 25,287,168 (в файле 25,287,168) | выход (2, 16, 4, 512), конечный: True
braindecode.models.EEGPT не проверен: ImportError("cannot import name 'EEGPT' from 'braindecode.models' (/usr/local/lib/python3.13/dist-packages/braindecode/models/__init__.py)")
figshare (оригинальный чекпоинт), ответ: 202
                                                check    ok
     ключи target_encoder braindecode = telecom-paris  True
                               формы и типы совпадают  True
                          значения совпадают побитово  True
                         chans_id braindecode = 0..61  True
strict-загрузка в EEGTransformer авторов (58 каналов)  True
strict-загрузка в EEGTransformer авторов (19 каналов)  True
                             braindecode.models.EEGPT  None
             figshare доступен из скрипта (ответ 202) False
EEGPT для дообучения: 25320228 параметров
готовые окна: NMT (70497, 19, 1024), Турция (17949, 19, 1024) | 21 с
label  abnormal  normal
loc
eval         90      95
train       325    1906
окон по 4 с: NMT train 44452 | NMT eval 26045 | Турция 17949 (на запись: медиана 150)
Турция: эпилепсия 50, здоровые 71
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 37755 (патология 0.14) | записей на выбор: 335 (6697 окон)
  проверка: эпоха 1 потери 0.701
  проверка: эпоха 2 потери 0.639
  проверка: эпоха 3 потери 0.594
  проверка: эпоха 4 потери 0.557
  проверка: эпоха 5 потери 0.520
  проверка: эпоха 6 потери 0.508
  проверка: эпоха 7 потери 0.500
  проверка: эпоха 8 потери 0.484
сходимость на подвыборке: есть
шаг 0.286 с | оценка эпохи 3.4 мин
  выбор lr 5e-05 запуск 0 | эпоха 1: потери 0.633 | отложенные NMT 0.782 | шагов с NaN/inf 0 | 219 с
  выбор lr 5e-05 запуск 0 | эпоха 2: потери 0.557 | отложенные NMT 0.846 | шагов с NaN/inf 0 | 456 с
  выбор lr 5e-05 запуск 0 | эпоха 3: потери 0.520 | отложенные NMT 0.845 | шагов с NaN/inf 0 | 693 с
  выбор lr 5e-05 запуск 0 | эпоха 4: потери 0.489 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 929 с
  выбор lr 5e-05 запуск 0 | эпоха 5: потери 0.477 | отложенные NMT 0.861 | шагов с NaN/inf 0 | 1165 с
  выбор lr 5e-05 запуск 0 | эпоха 6: потери 0.462 | отложенные NMT 0.874 | шагов с NaN/inf 0 | 1402 с
  выбор lr 5e-05 запуск 0 | эпоха 7: потери 0.453 | отложенные NMT 0.881 | шагов с NaN/inf 0 | 1638 с
  выбор lr 5e-05 запуск 0 | эпоха 8: потери 0.446 | отложенные NMT 0.878 | шагов с NaN/inf 0 | 1874 с
  выбор lr 5e-05 запуск 0 | эпоха 9: потери 0.441 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 2110 с
lr 5e-05: лучшая эпоха 7 из 9 (плато), отложенные NMT 0.881
ПРОПУСК lr 0.00015: не укладывается (нужно ~110 мин)
ПРОПУСК lr 0.0005: не укладывается (нужно ~110 мин)
выбрана скорость: 5e-05
eegpt запуск 0: эпоха 7 из 9 (плато) | NMT eval 0.893 | Турция 0.744 | медиана p здоровые/эпилепсия 0.332/0.560 | разных значений 119/121
ПРОПУСК запуск 1: не укладывается в 62 мин
ПРОПУСК запуск 2: не укладывается в 62 мин
всё обучение: 36.8 мин
model  lr  seed  best_epoch  epochs  stop   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr
eegpt 0.0     0           7       9 плато 0.881          0     0.893   0.744          0.332           0.56          119
                              модель n          NMT eval               Турция
          EEGPT дообученная, запуски 1     0.893 ± 0.000 0.744 ± 0.000 (0.74)
EEGPT дообученная, ансамбль запусков   0.893 (0.85-0.93)    0.744 (0.65-0.83)
Связь с амплитудой записи на Турции (Спирмен): 0.15
модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
 EEGPT                  Юден            0.451 NMT eval 185 0.79 (0.74-0.85)  0.80 (0.74-0.85) 0.75 (0.67-0.83) 0.86 (0.78-0.92) 0.74 (0.65-0.82)
 EEGPT                  Юден            0.451   Турция 121 0.66 (0.58-0.74)  0.66 (0.57-0.74) 0.58 (0.44-0.70) 0.62 (0.48-0.75) 0.69 (0.58-0.79)
 EEGPT чувствительность 80 %            0.502 NMT eval 185 0.81 (0.76-0.86)  0.81 (0.76-0.87) 0.79 (0.70-0.87) 0.83 (0.76-0.91) 0.79 (0.70-0.87)
 EEGPT чувствительность 80 %            0.502   Турция 121 0.69 (0.61-0.77)  0.68 (0.59-0.76) 0.64 (0.50-0.77) 0.58 (0.44-0.71) 0.77 (0.67-0.86)
всё время блокнота: 38.1 мин
```

## 6. Сеанс 2: запуски 1 и 2 (та же скорость 5·10⁻⁵, та же ранняя остановка)

Отдельный GPU-сеанс (версия 2 того же блокнота Kaggle: ячейки разделов 1–4 без проверки сходимости и выбора скорости, затем ячейка ниже). Kaggle выдал две T4, запуски шли параллельно. Аугментации переписаны без цикла по окнам: распределение то же, случайные числа другие.

In [ ]:
# --- второй сеанс GPU: запуски 1 и 2 с выбранной в первом сеансе скоростью 5·10⁻⁵ (та же ранняя остановка); при двух GPU — параллельно ---
import threading, traceback, contextlib
LR, SEEDS2, TRAIN_MIN2 = 5e-5, ([1] if SMOKE else [1, 2]), 40

def batch_input(W, g=None):
    """То же, что в разделе 1, но аугментации без цикла по окнам (то же распределение, быстрее)."""
    x = torch.from_numpy(np.ascontiguousarray(W)).to(DEV).float()
    if g is not None:
        m = torch.from_numpy(g.random(len(x)) < 0.5).to(DEV)
        x[m] = x[m][:, MIRROR]
    x = x - x.mean(1, keepdim=True)
    x = (x / (x.std(dim=(1, 2), keepdim=True) + 1e-6)).clamp(-CLIP, CLIP) * (STD_UV / 1000.0)
    x = x[:, ORDER]
    if g is not None:
        B, C, T = x.shape
        x = x + NOISE * (STD_UV / 1000.0) * torch.randn_like(x)
        k = np.where(g.random(B) < 0.5, g.integers(1, 3, B), 0)                    # 0, 1 или 2 зануляемых канала
        x[torch.from_numpy(np.argsort(g.random((B, C)), 1) < k[:, None]).to(DEV)] = 0
        sh = torch.from_numpy(g.integers(-2 * SFREQ, 2 * SFREQ + 1, B)).to(DEV)    # циклический сдвиг до 2 с
        x = x.gather(2, ((torch.arange(T, device=DEV)[None] - sh[:, None]) % T)[:, None, :].expand(B, C, T))
    return x

N_GPU = torch.cuda.device_count()
print("GPU:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
T2 = time.time(); DL2 = T2 + TRAIN_MIN2 * 60; RES2, LOCK = {}, threading.Lock()
_new = new_eegpt
def new_eegpt(seed):
    with LOCK:                      # torch.manual_seed общий для потоков
        return _new(seed)
def job(seed, dev):
    try:
        with (torch.cuda.device(dev) if DEV == "cuda" else contextlib.nullcontext()):
            t0 = time.time()
            model, info = train_run(seed, LR, DL2, tag=f"eegpt gpu{dev}")
            s_ev = predict(model, EV_X, EV_OFF); s_tr = predict(model, TRK_X, TRK_OFF); s_val = predict(model, VAL_X, VAL_OFF)
            RES2[seed] = dict(model="eegpt", lr=LR, seed=seed, best_epoch=info["best_ep"], epochs=info["epochs"], stop=info["stop"],
                              val=round(info["best_auc"], 3), hist=info["hist"], nan_steps=info["n_bad"],
                              nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr, s_val=s_val,
                              turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                              n_unique_tr=len(np.unique(s_tr)), minutes=round((time.time() - t0) / 60, 1))
            r = RES2[seed]
            print(f"eegpt запуск {seed}: эпоха {r['best_epoch']} из {r['epochs']} ({r['stop']}) | NMT eval {r['nmt_eval']:.3f} | Турция {r['turkey']:.3f} | "
                  f"медиана p здоровые/эпилепсия {r['turkey_med_p0']:.3f}/{r['turkey_med_p1']:.3f} | {r['minutes']} мин", flush=True)
            del model
            if DEV == "cuda": torch.cuda.empty_cache()
    except Exception:
        traceback.print_exc()
if N_GPU >= 2:
    th = [threading.Thread(target=job, args=(s, i)) for i, s in enumerate(SEEDS2)]
    for t in th: t.start()
    for t in th: t.join()
else:
    for s in SEEDS2:
        done = [r["minutes"] for r in RES2.values()]
        if done and time.time() + 0.8 * done[0] * 60 > DL2:
            print(f"ПРОПУСК запуск {s}: не укладывается в {TRAIN_MIN2} мин"); continue
        job(s, 0)
print(f"всё обучение второго сеанса: {(time.time()-T2)/60:.1f} мин")
runs2 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr", "s_val")} for r in RES2.values()])
runs2.to_csv(f"{OUT}/eegpt_finetune_runs_part2.csv", index=False)
print(runs2.drop(columns="hist").round(3).to_string(index=False))
d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": ev.y.to_numpy()})
d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr, "amp": TR_AMP})
d3 = pd.DataFrame({"dataset": "nmt_val", "id": list(val_m.recordname), "y": y_val})
for d, k in [(d1, "s_ev"), (d2, "s_tr"), (d3, "s_val")]:
    for s, r in sorted(RES2.items()): d[f"eegpt_seed{s}"] = r[k]
pd.concat([d1, d2, d3]).to_csv(f"{OUT}/eegpt_finetune_per_person_part2.csv", index=False)
print(f"всё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

Вывод Kaggle (сеанс 2):

```
каналы в порядке авторов: FP1 FP2 F7 F3 FZ F4 F8 T7 C3 CZ C4 T8 P7 P3 PZ P4 P8 O1 O2
их места среди 58: [0, 2, 5, 7, 9, 11, 13, 23, 25, 27, 29, 31, 41, 43, 45, 47, 49, 55, 57] | строки таблицы каналов: [0, 2, 7, 9, 11, 13, 15, 25, 27, 29, 31, 33, 43, 45, 47, 49, 51, 59, 61]
устройство: Tesla T4
braindecode: тензоров 103, параметров 25,287,230 | по частям: {'target_encoder': 102, 'chans_id': 1}
telecom-paris: тензоров 413, параметров 101,869,664 | по частям: {'reconstructor': 105, 'predictor': 104, 'encoder': 102, 'target_encoder': 102}
target_encoder: ключей 102 / 102 | ключи совпадают: True | формы и типы: True | значения побитово: True
telecom-paris: encoder против target_encoder (EMA-копия), макс. разница 0 — для дообучения берётся target_encoder, как у авторов
chans_id в файле braindecode = 0..61 (вся таблица каналов): True
код авторов, 58 каналов: strict=True — пропущено 0, лишних 0, параметров 25,287,168 (в файле 25,287,168) | выход (2, 16, 4, 512), конечный: True
код авторов, 19 каналов: strict=True — пропущено 0, лишних 0, параметров 25,287,168 (в файле 25,287,168) | выход (2, 16, 4, 512), конечный: True
braindecode.models.EEGPT не проверен: ImportError("cannot import name 'EEGPT' from 'braindecode.models' (/usr/local/lib/python3.13/dist-packages/braindecode/models/__init__.py)")
figshare (оригинальный чекпоинт), ответ: 202
                                                check    ok
     ключи target_encoder braindecode = telecom-paris  True
                               формы и типы совпадают  True
                          значения совпадают побитово  True
                         chans_id braindecode = 0..61  True
strict-загрузка в EEGTransformer авторов (58 каналов)  True
strict-загрузка в EEGTransformer авторов (19 каналов)  True
                             braindecode.models.EEGPT  None
             figshare доступен из скрипта (ответ 202) False
EEGPT для дообучения: 25320228 параметров
готовые окна: NMT (70497, 19, 1024), Турция (17949, 19, 1024) | 50 с
label  abnormal  normal
loc
eval         90      95
train       325    1906
окон по 4 с: NMT train 44452 | NMT eval 26045 | Турция 17949 (на запись: медиана 150)
Турция: эпилепсия 50, здоровые 71
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 37755 (патология 0.14) | записей на выбор: 335 (6697 окон)
GPU: 2 ['Tesla T4', 'Tesla T4']
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 1: потери 0.631 | отложенные NMT 0.785 | шагов с NaN/inf 0 | 193 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 1: потери 0.637 | отложенные NMT 0.801 | шагов с NaN/inf 0 | 205 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 2: потери 0.559 | отложенные NMT 0.844 | шагов с NaN/inf 0 | 397 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 2: потери 0.561 | отложенные NMT 0.840 | шагов с NaN/inf 0 | 420 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 3: потери 0.521 | отложенные NMT 0.856 | шагов с NaN/inf 0 | 601 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 3: потери 0.520 | отложенные NMT 0.856 | шагов с NaN/inf 0 | 634 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 4: потери 0.493 | отложенные NMT 0.836 | шагов с NaN/inf 0 | 806 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 4: потери 0.496 | отложенные NMT 0.855 | шагов с NaN/inf 0 | 847 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 5: потери 0.477 | отложенные NMT 0.871 | шагов с NaN/inf 0 | 1010 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 5: потери 0.474 | отложенные NMT 0.864 | шагов с NaN/inf 0 | 1061 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 6: потери 0.460 | отложенные NMT 0.872 | шагов с NaN/inf 0 | 1214 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 6: потери 0.461 | отложенные NMT 0.869 | шагов с NaN/inf 0 | 1274 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 7: потери 0.452 | отложенные NMT 0.873 | шагов с NaN/inf 0 | 1418 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 7: потери 0.454 | отложенные NMT 0.875 | шагов с NaN/inf 0 | 1487 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 8: потери 0.444 | отложенные NMT 0.878 | шагов с NaN/inf 0 | 1623 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 8: потери 0.447 | отложенные NMT 0.867 | шагов с NaN/inf 0 | 1701 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 9: потери 0.431 | отложенные NMT 0.880 | шагов с NaN/inf 0 | 1827 с
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 9: потери 0.437 | отложенные NMT 0.876 | шагов с NaN/inf 0 | 1914 с
  eegpt gpu1 lr 5e-05 запуск 2 | эпоха 10: потери 0.433 | отложенные NMT 0.879 | шагов с NaN/inf 0 | 2032 с
eegpt запуск 2: эпоха 9 из 10 (максимум эпох) | NMT eval 0.896 | Турция 0.733 | медиана p здоровые/эпилепсия 0.293/0.462 | 35.3 мин
  eegpt gpu0 lr 5e-05 запуск 1 | эпоха 10: потери 0.437 | отложенные NMT 0.877 | шагов с NaN/inf 0 | 2128 с
eegpt запуск 1: эпоха 10 из 10 (максимум эпох) | NMT eval 0.891 | Турция 0.768 | медиана p здоровые/эпилепсия 0.201/0.427 | 37.0 мин
всё обучение второго сеанса: 37.0 мин
model  lr  seed  best_epoch  epochs          stop   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr  minutes
eegpt 0.0     2           9      10 максимум эпох 0.880          0     0.896   0.733          0.293          0.462          119     35.3
eegpt 0.0     1          10      10 максимум эпох 0.877          0     0.891   0.768          0.201          0.427          119     37.0
всё время блокнота: 38.4 мин
```

## 7. Три запуска вместе (локально по csv: `eegpt_finetune_per_person_session1.csv` + `_session2.csv`)

Тот же код ансамбля, ДИ и порогов, что в разделе 5; порог — по отложенным 15 % NMT train (среднее вероятностей трёх запусков).

In [ ]:
# Три запуска вместе (локально, без GPU, по сохранённым вероятностям): ансамбль, ДИ, пороги — тот же код, что в разделе 5
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr
RES = "../results"
a = pd.read_csv(f"{RES}/eegpt_finetune_per_person_session1.csv")   # запуск 0 (сеанс 1)
b = pd.read_csv(f"{RES}/eegpt_finetune_per_person_session2.csv")   # запуски 1, 2 (сеанс 2)
pp = a.drop(columns=["eegpt_ens_rank"]).merge(b.drop(columns=["amp"]), on=["dataset", "id", "y"], how="inner")
assert len(pp) == len(a) == len(b)
seeds = [c for c in pp.columns if c.startswith("eegpt_seed")]
runs = pd.concat([pd.read_csv(f"{RES}/eegpt_finetune_runs_session1.csv"), pd.read_csv(f"{RES}/eegpt_finetune_runs_session2.csv")], ignore_index=True).sort_values("seed").reset_index(drop=True)
runs.to_csv(f"{RES}/eegpt_finetune_runs.csv", index=False)
print(runs[["seed", "lr", "best_epoch", "epochs", "stop", "val", "nan_steps", "nmt_eval", "turkey"]].round(3).to_string(index=False))

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
D = {ds: pp[pp.dataset == ds].reset_index(drop=True) for ds in ["nmt_eval", "turkey", "nmt_val"]}
ens = {ds: np.mean([rank(D[ds][c]) for c in seeds], 0) for ds in D}
y_ev, y_tr, y_val = (D[ds].y.to_numpy() for ds in ["nmt_eval", "turkey", "nmt_val"])
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "EEGPT дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "EEGPT дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens["nmt_eval"]), "Турция": fmt(y_tr, ens["turkey"])}])
summary.to_csv(f"{RES}/eegpt_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens["turkey"], D["turkey"].amp).correlation, 2))

p_val, p_ev, p_tr = (D[ds][seeds].mean(1).to_numpy() for ds in ["nmt_val", "nmt_eval", "turkey"])
fpr, tpr, thr = roc_curve(y_val, p_val)
THR = {"Юден": float(thr[np.argmax(tpr - fpr)]), "чувствительность 80 %": float(thr[np.argmax(tpr >= 0.8)])}
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}-{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
rows = []
for name, t in THR.items():
    for ds, y, s in [("NMT eval", y_ev, p_ev), ("Турция", y_tr, p_tr)]:
        m = metr_ci(y, s, t)
        rows.append({"модель": "EEGPT", "порог": name, "значение порога": round(t, 3), "набор": ds,
                     "примечание": "порог: отложенные 15 % NMT train" + ("" if ds == "NMT eval" else "; перенесён"), "n": len(y),
                     "accuracy": m["acc"], "сбаланс. accuracy": m["bacc"], "precision": m["prec"], "recall (чувств.)": m["rec"], "специфичность": m["spec"]})
thr_tab = pd.DataFrame(rows)
thr_tab.to_csv(f"{RES}/eegpt_threshold_metrics.csv", index=False)
print(thr_tab.drop(columns=["примечание"]).to_string(index=False))
for ds in ["nmt_eval", "turkey"]:
    pp.loc[pp.dataset == ds, "eegpt_ens_rank"] = ens[ds]
pp.to_csv(f"{RES}/eegpt_finetune_per_person.csv", index=False)

 seed  lr  best_epoch  epochs          stop   val  nan_steps  nmt_eval  turkey
    0 0.0           7       9         плато 0.881          0     0.893   0.744
    1 0.0          10      10 максимум эпох 0.877          0     0.891   0.768
    2 0.0           9      10 максимум эпох 0.880          0     0.896   0.733
                              модель n          NMT eval                             Турция
          EEGPT дообученная, запуски 3     0.894 ± 0.002 0.748 ± 0.018 (0.74 / 0.77 / 0.73)
EEGPT дообученная, ансамбль запусков   0.895 (0.85-0.94)                  0.755 (0.66-0.84)
Связь с амплитудой записи на Турции (Спирмен): 0.16
модель                 порог  значение порога    набор   n         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
 EEGPT                  Юден            0.380 NMT eval 185 0.79 (0.73-0.84)  0.79 (0.73-0.85) 0.74 (0.65-0.82) 0.88 (0.81-0.94) 0.71 (0.61-0.79)
 EEGPT                  Юден            0.380   Турция 121 0.68 (0

## 8. Сравнение с другими моделями, ансамбль с REVE, калибровка на месте (локально, без GPU, по сохранённым предсказаниям)

Запускать из папки блокнота: читает `results/eegpt_finetune_per_person.csv` и `results/eegdino_ensembles_per_person.csv` (общие записи NMT eval 184 + Турция 121 и ранги REVE, CBraMod, BIOT, LUNA, EEG-DINO, DeepEpilepsy). Калибровка — та же функция и те же 500 выборок (seed 0, тот же порядок людей), что в `luna_…` и `eegdino_turkey_local_threshold_calibration.csv`; совпадение проверяется по строкам прежних моделей.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import rankdata, spearmanr

RES = "../results"
eg = pd.read_csv(f"{RES}/eegpt_finetune_per_person.csv")
lu = pd.read_csv(f"{RES}/eegdino_ensembles_per_person.csv")   # общие записи и ранги всех прежних моделей, включая EEG-DINO
eg_cols = [c for c in eg.columns if c.startswith("eegpt_seed")]
m = lu.merge(eg[["dataset", "id", "y", "amp"] + eg_cols], on=["dataset", "id"], how="left", suffixes=("", "_e"))
assert m[eg_cols].notna().all().all() and (m.y == m.y_e).all() and len(m) == len(lu)
print({d: int((m.dataset == d).sum()) for d in ["nmt_eval", "turkey"]}, "общих записей")

rk = lambda s: rankdata(s) / len(s)
SC = {}
for ds in ["nmt_eval", "turkey"]:
    d = m[m.dataset == ds]
    s = {"EEGPT": np.mean([rk(d[c]) for c in eg_cols], 0), "REVE": rk(d.reve_rank), "CBraMod": rk(d.cbramod_rank),
         "BIOT": rk(d.biot_rank), "LUNA": rk(d.luna_rank), "EEG-DINO": rk(d.eegdino_rank), "DeepEpilepsy": rk(d.deepepilepsy_rank)}
    s["EEGPT + REVE"] = (s["EEGPT"] + s["REVE"]) / 2
    s["EEGPT + REVE + CBraMod"] = (s["EEGPT"] + s["REVE"] + s["CBraMod"]) / 3
    SC[ds] = (d.y.to_numpy(), s, d)

def boot(y, f, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(f(i))
    return np.percentile(v, [2.5, 97.5])

rows = []
for name in SC["turkey"][1]:
    row = {"модель / ансамбль (средние ранги)": name}
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]; lo, hi = boot(y, lambda i: roc_auc_score(y[i], s[name][i]))
        row[lab] = f"{roc_auc_score(y, s[name]):.3f} ({lo:.2f}-{hi:.2f})"
    rows.append(row)
summ = pd.DataFrame(rows)
print(summ.to_string(index=False))

print("\nПарный бутстреп разности ROC AUC (95% ДИ):")
diffs = []
for a, b in [("EEGPT", "REVE"), ("EEGPT", "CBraMod"), ("EEGPT", "BIOT"), ("EEGPT", "LUNA"), ("EEGPT", "EEG-DINO"), ("EEGPT", "DeepEpilepsy"),
             ("EEGPT + REVE", "REVE"), ("EEGPT + REVE", "EEGPT")]:
    for ds, lab in [("nmt_eval", "NMT eval"), ("turkey", "Турция")]:
        y, s, _ = SC[ds]
        f = lambda i: roc_auc_score(y[i], s[a][i]) - roc_auc_score(y[i], s[b][i])
        lo, hi = boot(y, f); dv = f(np.arange(len(y)))
        diffs.append({"разность": f"{a} − {b}", "набор": lab, "ΔROC AUC": round(dv, 3), "ДИ": f"{lo:+.3f}…{hi:+.3f}"})
        print(f"  {a} − {b} | {lab}: {dv:+.3f} ({lo:+.3f}…{hi:+.3f})")
y, s, d = SC["turkey"]
print("\nСпирмен на Турции: EEGPT–амплитуда", round(spearmanr(s["EEGPT"], d.amp).correlation, 2),
      "| " + " | ".join(f"EEGPT–{n} {spearmanr(s['EEGPT'], s[n]).correlation:.2f}" for n in ["REVE", "CBraMod", "BIOT", "LUNA", "EEG-DINO", "DeepEpilepsy"]))
print("амплитуда у других (Спирмен):", {n: round(float(spearmanr(s[n], d.amp).correlation), 2) for n in ["REVE", "CBraMod", "BIOT", "LUNA", "EEG-DINO", "DeepEpilepsy"]})
print("одна амплитуда как оценка, ROC AUC:", round(roc_auc_score(y, d.amp), 3))

# --- калибровка на месте: порог по k размеченным людям Турции (поровну классов, индекс Юдена), проверка на остальных; 500 повторов, одни и те же выборки для всех моделей ---
def calib(y, sc, k, reps=500, seed=0):
    r = np.random.default_rng(seed); i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]; M = {n: [] for n in sc}
    for _ in range(reps):
        c = np.concatenate([r.choice(i1, k // 2, replace=False), r.choice(i0, k // 2, replace=False)])
        rest = np.setdiff1d(np.arange(len(y)), c); yy = y[rest]
        for n, s in sc.items():
            fpr, tpr, th = roc_curve(y[c], s[c]); pr = s[rest] >= th[np.argmax(tpr - fpr)]
            tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
            rec, spec = tp / (tp + fn), tn / (tn + fp)
            M[n].append(dict(rec=rec, spec=spec, bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(yy)))
    out = []
    for n in sc:
        D = pd.DataFrame(M[n]); f = lambda c: f"{D[c].mean():.2f} ({np.nanpercentile(D[c], 2.5):.2f}–{np.nanpercentile(D[c], 97.5):.2f})"
        out.append({"калибровка": f"{k} чел.", "модель": n, "recall": f("rec"), "специфичность": f("spec"), "сбаланс. accuracy": f("bacc"),
                    "precision": f("prec"), "accuracy": f("acc")})
    return out
cal_models = ["EEGPT", "REVE", "CBraMod", "BIOT", "LUNA", "EEG-DINO", "DeepEpilepsy", "EEGPT + REVE"]
cal = pd.DataFrame([row for k in [10, 20, 30] for row in calib(y, {n: s[n] for n in cal_models}, k)])
old = pd.read_csv(f"{RES}/eegdino_turkey_local_threshold_calibration.csv")
chk = cal.merge(old, on=["калибровка", "модель"], suffixes=("", "_old"))
same = all((chk[c] == chk[c + "_old"]).all() for c in ["recall", "специфичность", "сбаланс. accuracy", "precision", "accuracy"])
print(f"\nвыборки совпадают с eegdino_turkey_local_threshold_calibration.csv: {same} (сверено {len(chk)} строк старых моделей)")
assert same
print("Калибровка на месте (среднее по 500 повторам, 2,5–97,5 % повторов):")
print(cal.to_string(index=False))

summ.to_csv(f"{RES}/eegpt_ensembles_summary.csv", index=False)
pd.DataFrame(diffs).to_csv(f"{RES}/eegpt_paired_differences.csv", index=False)
cal.to_csv(f"{RES}/eegpt_turkey_local_threshold_calibration.csv", index=False)
pp = []
for ds in ["nmt_eval", "turkey"]:
    yy, s, d = SC[ds]; t = d[["dataset", "id", "y"]].copy()
    for n, v in s.items(): t[n.replace(" + ", "_").replace(" ", "").lower() + "_rank"] = v
    pp.append(t)
pd.concat(pp).to_csv(f"{RES}/eegpt_ensembles_per_person.csv", index=False)

{'nmt_eval': 184, 'turkey': 121} общих записей
модель / ансамбль (средние ранги)          NMT eval            Турция
                            EEGPT 0.899 (0.85-0.94) 0.755 (0.66-0.84)
                             REVE 0.879 (0.83-0.92) 0.874 (0.80-0.93)
                          CBraMod 0.845 (0.79-0.90) 0.820 (0.74-0.88)
                             BIOT 0.798 (0.73-0.86) 0.821 (0.73-0.89)
                             LUNA 0.858 (0.80-0.91) 0.795 (0.71-0.87)
                         EEG-DINO 0.878 (0.83-0.92) 0.856 (0.79-0.91)
                     DeepEpilepsy 0.900 (0.85-0.94) 0.682 (0.59-0.77)
                     EEGPT + REVE 0.895 (0.85-0.93) 0.841 (0.76-0.91)
           EEGPT + REVE + CBraMod 0.887 (0.84-0.93) 0.852 (0.78-0.91)

Парный бутстреп разности ROC AUC (95% ДИ):
  EEGPT − REVE | NMT eval: +0.020 (-0.003…+0.045)
  EEGPT − REVE | Турция: -0.119 (-0.192…-0.049)
  EEGPT − CBraMod | NMT eval: +0.054 (+0.019…+0.092)
  EEGPT − CBraMod | Турция: -0.065 (-0.148…+0.019)
  EEGPT

## 9. Итоги

### Сверка весов (`results/eegpt_weights_check.csv`)
- `target_encoder` в HF `braindecode/eegpt-pretrained` и в HF `eeg-telecom-paris/eegpt-large-official`: 102 тензора; ключи, формы, типы и **значения совпадают побитово**.
- Строгая загрузка (`strict=True`) в `EEGTransformer` из кода авторов (коммит `ea7282f`): пропущено 0, лишних 0, 25 287 168 параметров — и для 58 каналов предобучения, и для наших 19.
- Не проверено: оригинал на figshare (из скрипта ответ 202 — проверка на робота) и класс `braindecode.models.EEGPT` (в braindecode 1.3.1 его нет). Веса сверены с независимой копией, этого достаточно.
- Подготовка сигнала: 256 Гц, окна 4 с (16 патчей по 64), мВ, средний референт, 19 каналов в порядке списка 58 каналов авторов (места 0, 2, 5, 7, 9, 11, 13, 23, 25, 27, 29, 31, 41, 43, 45, 47, 49, 55, 57). Отличие от авторов одно: каждое окно приводится к СКО 15 мкВ, потому что масштаб NMT неверный.

### ROC AUC (95% ДИ, бутстреп по людям; общие записи: NMT eval 184 + Турция 121)

| Модель / ансамбль (средние ранги) | NMT eval | Турция |
|---|---|---|
| **EEGPT (3 запуска)** | **0,899 (0,85–0,94)** | 0,755 (0,66–0,84) |
| REVE | 0,879 (0,83–0,92) | **0,874 (0,80–0,93)** |
| EEG-DINO | 0,878 (0,83–0,92) | 0,856 (0,79–0,91) |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| BIOT | 0,798 (0,73–0,86) | 0,821 (0,73–0,89) |
| LUNA | 0,858 (0,80–0,91) | 0,795 (0,71–0,87) |
| DeepEpilepsy | 0,900 (0,85–0,94) | 0,682 (0,59–0,77) |
| EEGPT + REVE | 0,895 (0,85–0,93) | 0,841 (0,76–0,91) |
| EEGPT + REVE + CBraMod | 0,887 (0,84–0,93) | 0,852 (0,78–0,91) |

Запуски EEGPT: NMT eval 0,893 / 0,891 / 0,896, Турция 0,744 / 0,768 / 0,733 (0,748 ± 0,018). На всех 185 записях NMT eval ансамбль даёт 0,895 (0,85–0,94).

Парный бутстреп разности ROC AUC (Турция; NMT eval):
- **EEGPT − REVE: −0,119 (−0,192…−0,049)** — на переносе значимо хуже; NMT eval +0,020 (−0,003…+0,045).
- **EEGPT − EEG-DINO: −0,101 (−0,178…−0,027)** — значимо хуже; NMT eval +0,021 (не значимо).
- EEGPT − CBraMod: −0,065 (−0,148…+0,019); NMT eval **+0,054 (+0,019…+0,092)**.
- EEGPT − BIOT: −0,066 (не значимо); NMT eval **+0,102 (+0,051…+0,158)**.
- EEGPT − LUNA: −0,040 (не значимо); NMT eval +0,042 (−0,003…+0,090).
- EEGPT − DeepEpilepsy: +0,074 (−0,020…+0,159); NMT eval 0,000.
- EEGPT + REVE − REVE: Турция −0,033 (−0,068…+0,002); NMT eval **+0,016 (+0,003…+0,030)**.

### Пороговые метрики (среднее вероятностей 3 запусков; порог по отложенным 15 % NMT train, на Турцию перенесён тот же)

| Порог | Набор | Accuracy | Сбаланс. accuracy | Precision | Recall | Специфичность |
|---|---|---|---|---|---|---|
| Юден (0,380) | NMT eval (185) | 0,79 (0,73–0,84) | 0,79 (0,73–0,85) | 0,74 (0,65–0,82) | 0,88 (0,81–0,94) | 0,71 (0,61–0,79) |
| то же, перенесён | Турция (121) | 0,68 (0,60–0,76) | 0,67 (0,58–0,75) | 0,60 (0,46–0,73) | 0,64 (0,50–0,77) | 0,70 (0,59–0,81) |
| чувств. 80 % (0,482) | NMT eval | 0,83 (0,77–0,88) | 0,83 (0,77–0,88) | 0,80 (0,72–0,88) | 0,86 (0,78–0,92) | 0,80 (0,72–0,88) |
| то же, перенесён | Турция | 0,75 (0,67–0,83) | 0,72 (0,64–0,80) | 0,79 (0,64–0,92) | 0,54 (0,40–0,68) | 0,90 (0,83–0,96) |

### Калибровка на месте (Турция: порог по k людям поровну классов, Юден; 500 повторов, те же выборки, что у LUNA и EEG-DINO)

Сбалансированная accuracy:

| k | EEGPT | REVE | EEG-DINO | BIOT | CBraMod | LUNA | DeepEpilepsy | EEGPT + REVE |
|---|---|---|---|---|---|---|---|---|
| 10 | 0,65 (0,53–0,72) | 0,75 | 0,72 | 0,71 | 0,69 | 0,69 | 0,59 | 0,74 |
| 20 | **0,67 (0,58–0,73)** | 0,77 | 0,74 | 0,73 | 0,69 | 0,69 | 0,60 | 0,75 |
| 30 | 0,67 (0,60–0,74) | 0,77 | 0,74 | 0,73 | 0,70 | 0,69 | 0,61 | 0,76 |

При 20 людях у EEGPT: recall 0,54, специфичность 0,79, precision 0,70, accuracy 0,69. Полная таблица — `results/eegpt_turkey_local_threshold_calibration.csv`.

## 10. Выводы

1. **EEGPT лучше всех внутри NMT, но переносится хуже остальных трансформеров.** NMT eval 0,899 — наравне с DeepEpilepsy (0,900); CBraMod (+0,05) и BIOT (+0,10) она обходит значимо. На Турции 0,755 (0,66–0,84): значимо ниже REVE (−0,12) и EEG-DINO (−0,10); ниже CBraMod, BIOT и LUNA на 0,04–0,07, но эти разности не значимы. Картина та же, что у DeepEpilepsy: чем лучше модель подстраивается под NMT, тем хуже переносится.
2. **В ансамбле с REVE пользы на переносе нет**: EEGPT + REVE 0,841 против 0,874 у одной REVE (−0,03, не значимо); на NMT eval +0,016 (значимо, но мало).
3. **Перенесённый порог работает заметно лучше, чем у других моделей**: на Турции чувствительность 0,64 при специфичности 0,70 (порог Юдена); у REVE, CBraMod, BIOT и LUNA было 0,0–0,16. Вероятности EEGPT меньше «съезжают» между больницами. Но при калибровке на месте по 20 людям сбалансированная accuracy лишь 0,67 — ниже REVE (0,77) и EEG-DINO (0,74): мешает слабое ранжирование.
4. **Связь с амплитудой слабая**: Спирмен +0,16 (REVE +0,31, BIOT +0,25, EEG-DINO +0,24, LUNA +0,19, CBraMod +0,12). Одна амплитуда записи даёт AUC 0,76 — больше, чем EEGPT на Турции.
5. **Плато.** Запуск 0 остановился по правилу (лучшая эпоха 7 из 9); отложенный AUC вырос 0,86 → 0,88 к эпохе 4–7. Запуски 1 и 2 дошли до потолка в 10 эпох (лучшие эпохи 10 и 9), но с 7-й эпохи кривая плоская: 0,875 / 0,867 / 0,876 / 0,877 и 0,873 / 0,878 / 0,880 / 0,879. Ошибка LUNA (рост до последней эпохи на 0,01–0,02 за эпоху) не повторилась; заметного выигрыша от лишних эпох ждать не стоит.
6. **Проверена только одна скорость обучения (5·10⁻⁵), а не 2–3, как планировалось.** Эпоха заняла 3,9 мин, и защита по времени пропустила 1,5·10⁻⁴ и 5·10⁻⁴. Так что 5·10⁻⁵ — не выбор по сетке, а единственное проверенное значение. Для сравнения: у LUNA значение авторов 5·10⁻⁴ было худшим из трёх.
7. **Плюс для TUEP**: EEGPT предобучена на PhysioNetMI, HGD, TSU, SEED и M3CV — **TUH она не видела**. На TUEP её можно проверять без исключения пациентов, в отличие от REVE, CBraMod, LUNA и LaBraM.
8. **Ограничения**: 3 запуска; в Турции 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи. Отложенные 15 % NMT train делились тем же правилом, что у LUNA, но в train прочиталась на одну запись больше (2231 против 2230), поэтому отложенная часть может немного отличаться. Запуски 1–2 шли с векторными аугментациями (распределение то же). Дообученные веса не сохранялись.

**Затраты GPU:** сеанс 1 — 38,1 мин работы блокнота, сеанс 2 — 38,4 мин (две T4 параллельно). Квота Kaggle за оба сеанса: 19,25 → 21,28 ч; во время сеанса 1 параллельно шёл EEG-DINO, так что на EEGPT приходится **≈1,3 ч GPU** (бюджет 1,5 ч и лимит недели 25 ч соблюдены). Чтение данных (23 мин) и три проверки на процессоре Kaggle GPU не тратили.

Файлы: `results/eegpt_weights_check.csv`, `eegpt_lr_choice.csv`, `eegpt_finetune_{runs,summary,per_person}.csv` (исходные выгрузки — с суффиксами `_session1`, `_session2`), `eegpt_threshold_metrics.csv`, `eegpt_ensembles_{summary,per_person}.csv`, `eegpt_paired_differences.csv`, `eegpt_turkey_local_threshold_calibration.csv`.